# Preprocessing Melhorado - MIQR-CC Dataset

Este notebook estende o pipeline de preprocessing da baseline original com:
- **CLAHE** (Contrast Limited Adaptive Histogram Equalization)
- **Padding com aspect ratio preservado** antes do resize
- **Normalização** de intensidade
- **Data Augmentation** para treino
- **Pipeline unificado** pronto a integrar com PyTorch DataLoader

A lógica de segmentação por linhas verticais (Canny + Hough) é mantida da baseline:
> Martins et al., *MIQR-CC Dataset*, https://github.com/monicaccmartins/MIQR-CC-Dataset

O split treino/validação/teste respeita o split original da baseline para comparação justa.

In [ ]:
import os
import math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

---
## 1. Segmentação por linhas verticais (baseline original)

Mantida sem alterações da baseline. Deteta linhas verticais com Canny + Hough Transform.

In [ ]:
def detect_vertical_lines(image, canny1=30, canny2=150, hough_threshold=120,
                           min_line_length=70, max_line_gap=4, angle_threshold_deg=10):
    """Deteta linhas verticais via Canny + HoughLinesP. (baseline original)"""
    edges = cv2.Canny(image, canny1, canny2)
    lines = cv2.HoughLinesP(
        edges, rho=1, theta=np.pi / 180,
        threshold=hough_threshold,
        minLineLength=min_line_length,
        maxLineGap=max_line_gap
    )
    vertical_lines = []
    if lines is not None:
        for l in lines:
            x1, y1, x2, y2 = l[0]
            angle = abs(math.degrees(math.atan2(y2 - y1, x2 - x1)))
            if abs(angle - 90) <= angle_threshold_deg:
                vertical_lines.append(l)
    return edges, vertical_lines


def segment_image_by_vertical_lines(image, lines, min_width=30):
    """Corta a imagem nos x das linhas verticais detetadas. (baseline original)"""
    if not lines:
        return [image]
    lines = sorted(lines, key=lambda x: x[0][0])
    x_positions = [0] + [l[0][0] for l in lines] + [image.shape[1]]
    segments = []
    for i in range(len(x_positions) - 1):
        x_start, x_end = x_positions[i], x_positions[i + 1]
        if x_end - x_start >= min_width:
            segments.append(image[:, x_start:x_end])
    return segments if segments else [image]


def is_image_valid(image, min_width=212, black_percentile=90, contrast_threshold=10):
    """Valida segmento por largura, escuridão e contraste. (baseline original)"""
    h, w = image.shape
    if w < min_width:
        return False
    if np.percentile(image, black_percentile) < 1:
        return False
    if image.max() - image.min() < contrast_threshold:
        return False
    return True

---
## 2. Melhorias: CLAHE + Padding + Normalização

In [ ]:
def apply_clahe(image, clip_limit=2.0, tile_grid=(8, 8)):
    """
    Aplica CLAHE para realçar contraste local.
    Especialmente útil em fluoroscopia onde cálculos e estenoses têm contraste subtil.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(image)


def pad_to_square(image):
    """
    Padding preto para tornar a imagem quadrada,
    preservando o aspect ratio original.
    """
    h, w = image.shape[:2]
    size   = max(h, w)
    top    = (size - h) // 2
    bottom = size - h - top
    left   = (size - w) // 2
    right  = size - w - left
    return cv2.copyMakeBorder(image, top, bottom, left, right,
                               borderType=cv2.BORDER_CONSTANT, value=0)


def resize_image(image, target_size=300):
    return cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_AREA)


def normalize_image(image):
    img  = image.astype(np.float32) / 255.0
    mean = 0.485
    std  = 0.229
    return (img - mean) / std

---
## 3. Pipeline unificado

In [ ]:
def preprocess_image(image_path, target_size=300, apply_norm=True):
    """
    Pipeline completo:
      1. Lê imagem em grayscale
      2. Segmentação por linhas verticais (baseline)
      3. Seleciona segmento válido mais largo
      4. CLAHE
      5. Padding + resize
      6. Normalização (opcional)
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    edges, lines = detect_vertical_lines(img)
    segments     = segment_image_by_vertical_lines(img, lines)
    valid_segs   = [s for s in segments if is_image_valid(s)]
    seg = max(valid_segs, key=lambda s: s.shape[1]) if valid_segs else img

    seg = apply_clahe(seg)
    seg = pad_to_square(seg)
    seg = resize_image(seg, target_size)

    if apply_norm:
        seg = normalize_image(seg)

    return seg

---
## 4. Visualização comparativa

In [ ]:
def visualize_preprocessing(image_path, target_size=300):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f'Erro ao abrir: {image_path}')
        return

    edges, lines = detect_vertical_lines(img)
    segments     = segment_image_by_vertical_lines(img, lines)
    valid_segs   = [s for s in segments if is_image_valid(s)]
    seg_baseline = max(valid_segs, key=lambda s: s.shape[1]) if valid_segs else img

    seg_clahe   = apply_clahe(seg_baseline)
    seg_padded  = pad_to_square(seg_clahe)
    seg_resized = resize_image(seg_padded, target_size)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, title, image in zip(axes,
        ['Original', 'Baseline (crop)', '+ CLAHE', '+ Padding + Resize'],
        [img, seg_baseline, seg_clahe, seg_resized]):
        ax.imshow(image, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    plt.suptitle(os.path.basename(image_path), fontsize=10)
    plt.tight_layout()
    plt.show()

---
## 5. Gerar dataset_processed mantendo o split da baseline

Aplica o pipeline melhorado a cada imagem, mantendo a estrutura
`train/val/test` e as classes do split original da baseline.
Isto garante comparação justa com os resultados do artigo.

In [ ]:
# ============================================================
# CONFIGURAÇÃO — ajusta os paths ao teu ambiente
# ============================================================
DATASET_BASELINE = './dataset'           # pasta com train/val/test da baseline
DATASET_OUTPUT   = './dataset_processed' # destino com as imagens melhoradas
TARGET_SIZE      = 300                   # EfficientNet-B3; usa 224 para ResNet-50
# ============================================================

errors   = []
total_ok = 0

for split in ['train', 'val', 'test']:
    split_dir = os.path.join(DATASET_BASELINE, split)
    if not os.path.exists(split_dir):
        print(f'Pasta não encontrada: {split_dir}')
        continue

    classes = os.listdir(split_dir)
    print(f'\n[{split}] — {len(classes)} classes')

    for class_name in classes:
        class_src  = os.path.join(split_dir, class_name)
        class_dest = os.path.join(DATASET_OUTPUT, split, class_name)
        os.makedirs(class_dest, exist_ok=True)

        images = [f for f in os.listdir(class_src) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for img_name in tqdm(images, desc=f'  {class_name}', leave=False):
            src  = os.path.join(class_src, img_name)
            dest = os.path.join(class_dest, img_name)

            if os.path.exists(dest):  # skip se já processado
                total_ok += 1
                continue

            img = preprocess_image(src, target_size=TARGET_SIZE, apply_norm=False)
            if img is None:
                errors.append(src)
                continue

            Image.fromarray(img.astype(np.uint8)).save(dest)
            total_ok += 1

print(f'\nProcessadas com sucesso: {total_ok}')
print(f'Erros: {len(errors)}')
if errors:
    print('Ficheiros com erro:')
    for e in errors:
        print(f'  {e}')

In [ ]:
# Verifica distribuição final do dataset_processed
print('Distribuição do dataset_processed:\n')
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(DATASET_OUTPUT, split)
    print(f'[{split}]')
    total = 0
    for class_name in sorted(os.listdir(split_dir)):
        n = len(os.listdir(os.path.join(split_dir, class_name)))
        print(f'  {class_name:20s}: {n}')
        total += n
    print(f'  Total: {total}\n')

In [ ]:
# Visualiza exemplos do preprocessing em imagens do treino
import random

for class_name in os.listdir(os.path.join(DATASET_BASELINE, 'train')):
    class_dir = os.path.join(DATASET_BASELINE, 'train', class_name)
    images    = [f for f in os.listdir(class_dir) if f.endswith('.png')]
    sample    = random.choice(images)
    print(f'--- {class_name} ---')
    visualize_preprocessing(os.path.join(class_dir, sample), target_size=TARGET_SIZE)